In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns
import copy
import pickle
import shap
from models import *
from utilities import *
import warnings
warnings.filterwarnings('ignore')

In [47]:
##########################################################################################################

In [48]:
import tensorflow as tf
# tf.compat.v1.disable_v2_behavior()

In [49]:
# shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
# shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough

In [50]:
################### data import ###############

In [51]:
data_dir = '../results/data/'
data_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_dy_filter.pkl')
ba_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_ba_filter.pkl')
data_copy['Id_encoded'], uniques = pd.factorize(data_copy['SCCRIP_ID'])
data_copy = data_copy.fillna(0)
mapping_dict = {value: index for index, value in enumerate(uniques)}

In [52]:
hu = read_sas('../data/Apply_MI_Zheng_2023_01/hu.sas7bdat')

In [53]:
bmt = read_sas('../data/Apply_MI_Zheng_2023_01/othertreatment_summary.sas7bdat')
bmt = bmt[['SCCRIP_ID', 'BMT']]
bmt_ids = set(bmt[bmt['BMT'] == 1].SCCRIP_ID)
data_copy_filter = data_copy[~data_copy["SCCRIP_ID"].isin(bmt_ids)]
len(data_copy_filter.SCCRIP_ID.unique())

1264

In [54]:
data_copy_nohu_f = data_copy_filter[data_copy_filter['HU_status']==0]
data_copy_hu_f = data_copy_filter[data_copy_filter['HU_status']!=0]

In [55]:
data_copy_nohu = data_copy[data_copy['HU_status']==0]
data_copy_hu = data_copy[data_copy['HU_status']!=0]

In [56]:
ba_copy['Id_encoded'] = ba_copy['SCCRIP_ID'].map(mapping_dict)
ba_copy = ba_copy[ba_copy['Id_encoded'].notna()]
ba_copy = ba_copy.drop_duplicates(subset='SCCRIP_ID', keep='last')
ba_copy.drop('SCCRIP_ID', inplace=True, axis=1)
ba_copy = ba_copy.fillna(0)

In [57]:
feature_dy = data_copy_nohu_f.drop(['HU_status','Id_encoded','SCCRIP_ID', 'date'],axis=1).columns
feature_ba = ba_copy.drop(['Id_encoded'], axis=1).columns

In [58]:
len(feature_dy), len(feature_ba)

(786, 23)

In [59]:
################ data process ##################

In [60]:
seq_length = 5

In [61]:
r2 = []
shap_impact = []
sequences, seq_ba, labels, date,  y_target = create_dataset(data_copy_nohu_f, ba_copy, seq_length, mapping_dict)
sequences_copy, seq_ba_copy, labels_copy, date_copy, y_target_copy = shuffle_dataset(sequences, seq_ba, labels, date, y_target)
X, y, X_bg, dt, ls, seq_map = create_sequences_with_background(sequences_copy, labels_copy, seq_ba_copy, date_copy,y_target_copy, seq_length)
X_normalized, X_bg_normalized = normalize_X(X, X_bg)

while len(shap_impact) < 20:
    print(len(shap_impact)) 
    (X_train_nohu,X_val_nohu,X_test_nohu,
                X_bg_train_nohu,X_bg_val_nohu,X_bg_test_nohu,
                dt_train_nohu, dt_val_nohu, dt_test_nohu, 
                y_train_nohu,y_val_nohu,y_test_nohu,
                ls_train_nohu, ls_val_nohu, ls_test_nohu,
                seq_map_train_nohu, seq_map_val_nohu, seq_map_test_nohu) = split_dataset(X_normalized, X_bg_normalized,dt, y, ls, seq_map)
    
    combined_model_l = CombinedLSTMModel(seq_length=seq_length, n_features=int(X_train_nohu.shape[2]), bg_features=int(X_bg_train_nohu.shape[2]), summary=False)
    
    history = combined_model_l.train(
    X_train_nohu, X_bg_train_nohu, dt_train_nohu, y_train_nohu,
    X_val_nohu,   X_bg_val_nohu,   dt_val_nohu,   y_val_nohu,
    epochs=20, batch_size=64,
    checkpoint_path="../results/models/C_best2_loop.h5")
    y_pred_nohu = combined_model_l.predict(X_test_nohu, X_bg_test_nohu, dt_test_nohu)
    r2_nohu = r2_score(y_test_nohu.flatten(), y_pred_nohu.flatten())
    if r2_nohu > 0:
        r2.append(r2_nohu)

        explainer = shap.GradientExplainer(combined_model_l.model, [X_test_nohu, X_bg_test_nohu, dt_test_nohu])
        shap_values = explainer.shap_values([X_test_nohu, X_bg_test_nohu, dt_test_nohu])

        shap_val_d = []
        train_d = []

        for i in range (5):
            shap_val_d.append(pd.DataFrame(shap_values[0][:,i,:,0], columns=feature_dy))
            train_d.append(pd.DataFrame(X_test_nohu[:,i,:],columns=feature_dy))
            
        shap_val_b = pd.DataFrame(shap_values[1][:,0,:,0], columns=feature_ba)
        train_b = pd.DataFrame(X_bg_test_nohu[:,0,:], columns=feature_ba)

        impact_dy = []

        for i in range(5):
            im = ABS_SHAP(shap_val_d[i], train_d[i])
            impact_dy.append(im)
            
        impact_ba = ABS_SHAP(shap_val_b, train_b)

        shap_impact.append((impact_dy, impact_ba))

0
Epoch 1/20
68/68 [==============================] - ETA: 0s - loss: 87.1315 - mean_absolute_error: 5.2921
Epoch 1: val_loss improved from inf to 53.05978, saving model to ../results/models/C_best2_loop.h5
68/68 [==============================] - 5s 22ms/step - loss: 87.1315 - mean_absolute_error: 5.2921 - val_loss: 53.0598 - val_mean_absolute_error: 4.4249
Epoch 2/20
64/68 [===========================>..] - ETA: 0s - loss: 67.5142 - mean_absolute_error: 4.8118
Epoch 2: val_loss improved from 53.05978 to 36.34697, saving model to ../results/models/C_best2_loop.h5
68/68 [==============================] - 1s 11ms/step - loss: 65.9492 - mean_absolute_error: 4.7463 - val_loss: 36.3470 - val_mean_absolute_error: 3.6040
Epoch 3/20
63/68 [==========================>...] - ETA: 0s - loss: 39.5223 - mean_absolute_error: 3.4752
Epoch 3: val_loss improved from 36.34697 to 19.57477, saving model to ../results/models/C_best2_loop.h5
68/68 [==============================] - 1s 11ms/step - loss: 38.

In [62]:
with open('../results/data/cmodel/shap_impact_list_noBMTv4.pkl','wb') as f:
    pickle.dump(shap_impact,f)

In [63]:
with open('../results/data/cmodel/r2_Cmodel_noBMT.pkl','wb') as f:
    pickle.dump(r2,f)

In [64]:
############################################